# Train & evaluate — StackLearn similar-question recommender

Run **data_preprocessing.ipynb** first: it writes the clean catalog to `dataset/questions_clean.csv`
and the shared config to `outputs/config.json`. This notebook loads the clean catalog,
splits it temporally, trains the candidate models (TF-IDF, sentence embeddings),
evaluates them on the held-out test split, and persists the artifacts `app.py` loads
from `outputs/`.


In [ ]:
!pip install -q scikit-learn sentence-transformers tqdm


## Config — read from `outputs/config.json` (written by data_preprocessing.ipynb)

In [ ]:
import os, json
import numpy as np
import pandas as pd

CONFIG = json.load(open("outputs/config.json"))  # data_preprocessing.ipynb must have run first
out = CONFIG["output_dir"]
os.makedirs(out, exist_ok=True)

LOG = []
def log(msg):
    print(msg)
    LOG.append(str(msg))


## 0. Load the clean dataset + 1. Labels (tag vocabulary, multi-hot matrix)

In [ ]:
from collections import Counter

df = pd.read_csv(CONFIG["clean_path"])
log(f"loaded {len(df)} rows from {CONFIG['clean_path']}")
df["question_id"] = df["question_id"].astype(str)
df["creation_date"] = pd.to_datetime(df["creation_date"], utc=True)
for c in ["is_answered", "has_accepted_answer"]:
    df[c] = df[c].astype(str).str.lower().map({"true": True, "false": False})
for c in ["score", "view_count", "answer_count"]:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# label side: AI/ML tag vocabulary + multi-hot matrix (n_questions x n_tags)
domain = set(CONFIG["domain_tags"])
df["domain_tag_list"] = df["tags"].fillna("").str.split("|").map(lambda ts: sorted(set(ts) & domain))
counts = Counter(t for ts in df["domain_tag_list"] for t in ts)
vocab = sorted(t for t, c in counts.items() if c >= CONFIG["min_tag_freq"])
tag2idx = {t: i for i, t in enumerate(vocab)}
log(f"tag vocabulary: {len(vocab)} tags with freq >= {CONFIG['min_tag_freq']}")
log("top tags: " + str(counts.most_common(10)))
Y = np.zeros((len(df), len(vocab)), dtype=np.int8)
for i, ts in enumerate(df["domain_tag_list"]):
    for t in ts:
        if t in tag2idx:
            Y[i, tag2idx[t]] = 1
log(f"label matrix Y: {Y.shape}, avg tags/question: {Y.sum(axis=1).mean():.2f}")


## 2. Temporal train/val/test split (leak-free)
Split by `creation_date`: train ≤ train_end, val ≤ val_end, test after. No random shuffle — the
data is time-ordered.


In [12]:
train_end = pd.Timestamp(CONFIG["split_dates"]["train_end"], tz="UTC")
val_end = pd.Timestamp(CONFIG["split_dates"]["val_end"], tz="UTC")
train = df[df["creation_date"] <= train_end].reset_index(drop=True)
val   = df[(df["creation_date"] > train_end) & (df["creation_date"] <= val_end)].reset_index(drop=True)
test  = df[df["creation_date"] > val_end].reset_index(drop=True)
log(f"split: train={len(train)}, val={len(val)}, test={len(test)}")
assert len(test) == 0 or (train["creation_date"].max() < val["creation_date"].min() <= test["creation_date"].min())
assert not (set(train["question_id"]) & set(test["question_id"]))

split: train=35867, val=118, test=83


## 3. Hyperparameter tuning — small grid over TF-IDF settings, scored on val (fit on train only)
Queries = val questions, catalog = train (same protocol as the test evaluation). The last
grid row is also the stopword-removal ablation (cf. text preprocessing in the NLP lecture).
Pick the best setting and use it for the final vectorizer in the next cell.


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.metrics.pairwise import cosine_similarity

def make_vectorizer(word_ngrams, char_ngrams, min_df, max_features, stop_words):
    w = TfidfVectorizer(analyzer="word", ngram_range=word_ngrams, min_df=min_df,
                        max_features=max_features, sublinear_tf=True, stop_words=stop_words)
    c = TfidfVectorizer(analyzer="char_wb", ngram_range=char_ngrams, min_df=min_df,
                        max_features=max_features, sublinear_tf=True)
    return FeatureUnion([("word", w), ("char", c)])

def val_precision_at_5(vectorizer, k=5):
    Xtr = vectorizer.fit_transform(train["text_all"])
    Xq = vectorizer.transform(val["text_all"])
    sims = cosine_similarity(Xq, Xtr)
    hits = 0
    for i in range(len(val)):
        top = np.argsort(-sims[i])[:k]
        true_tags = set(val["domain_tag_list"].iloc[i])
        hits += sum(1 for j in top if set(train["domain_tag_list"].iloc[j]) & true_tags)
    return hits / (k * len(val))

grid = [
    {"word_ngrams": (1, 1), "char_ngrams": (3, 5), "min_df": 2, "max_features": 100000, "stop_words": None},
    {"word_ngrams": (1, 2), "char_ngrams": (3, 5), "min_df": 2, "max_features": 100000, "stop_words": None},
    {"word_ngrams": (1, 2), "char_ngrams": (3, 5), "min_df": 3, "max_features": 50000, "stop_words": None},
    {"word_ngrams": (1, 2), "char_ngrams": (3, 5), "min_df": 2, "max_features": 100000, "stop_words": "english"},
]
best_p, best_tfidf_params = -1.0, None
for g in grid:
    p5 = val_precision_at_5(make_vectorizer(**g))
    log(f"grid {g} -> val precision@5 = {p5:.3f}")
    if p5 > best_p:
        best_p, best_tfidf_params = p5, g
log(f"best TF-IDF params (val precision@5={best_p:.3f}): {best_tfidf_params}")


grid {'word_ngrams': (1, 1), 'char_ngrams': (3, 5), 'min_df': 2, 'max_features': 100000, 'stop_words': None} -> val precision@5 = 0.959
grid {'word_ngrams': (1, 2), 'char_ngrams': (3, 5), 'min_df': 2, 'max_features': 100000, 'stop_words': None} -> val precision@5 = 0.963
grid {'word_ngrams': (1, 2), 'char_ngrams': (3, 5), 'min_df': 3, 'max_features': 50000, 'stop_words': None} -> val precision@5 = 0.963
grid {'word_ngrams': (1, 2), 'char_ngrams': (3, 5), 'min_df': 2, 'max_features': 100000, 'stop_words': 'english'} -> val precision@5 = 0.964


## 4. TF-IDF vectorization with the best params from tuning (fit on train only)
Word n-grams (1–2) + char n-grams (3–5) over `text_all` — char n-grams catch code tokens. The
fitted vectorizer is persisted and reused at inference; it is never refit on val/test.


In [13]:
vectorizer = make_vectorizer(**best_tfidf_params)
X_train = vectorizer.fit_transform(train["text_all"])
X_val = vectorizer.transform(val["text_all"])
X_test = vectorizer.transform(test["text_all"])
log(f"TF-IDF final (best params): train {X_train.shape}, val {X_val.shape}, test {X_test.shape}")


TF-IDF: train (35867, 200000), val (118, 200000), test (83, 200000) (vocab 200,000)


## 5. Evaluation — precision@k / recall@k / MRR on the test split
A retrieved question counts as relevant if it shares ≥ 1 domain tag with the query. Catalog =
train split (the app serves the train catalog in this pilot).


In [14]:
from sklearn.metrics.pairwise import cosine_similarity

n_eval = min(CONFIG["max_eval_queries"], len(test))
Xq = X_test[:n_eval]
sims = cosine_similarity(Xq, X_train)          # queries x catalog
train_tags = train["domain_tag_list"].tolist()

def evaluate(k):
    prec, rec, rr = [], [], []
    for i in range(n_eval):
        top = np.argsort(-sims[i])[:k]
        true_tags = set(test["domain_tag_list"].iloc[i])
        hits = sum(1 for j in top if set(train_tags[j]) & true_tags)
        prec.append(hits / k)
        rec.append(hits / max(1, sum(1 for tags in train_tags if set(tags) & true_tags)))  # relevant = catalog items sharing a tag
        for rank, j in enumerate(top, start=1):
            if set(train_tags[j]) & true_tags:
                rr.append(1 / rank)
                break
        else:
            rr.append(0.0)
    return np.mean(prec), np.mean(rec), np.mean(rr)

for k in CONFIG["top_k"]:
    p, r, mrr = evaluate(k)
    log(f"test n={n_eval}: precision@{k}={p:.3f} recall@{k}={r:.3f} MRR={mrr:.3f}")

# qualitative sanity check: top-5 for the first 3 test questions
for i in range(min(3, n_eval)):
    top = np.argsort(-sims[i])[:5]
    log(f"QUERY: {test['title'].iloc[i][:90]}")
    for j in top:
        log(f"  {sims[i, j]:.3f}  {train['title'].iloc[j][:80]}  [{', '.join(train_tags[j])}]")

test n=83: precision@5=0.981 recall@5=0.000 MRR=0.988
test n=83: precision@10=0.976 recall@10=0.000 MRR=0.988
QUERY: Buggy implementation of Knuth's Mastermind algorithm in Python
  0.164  Performance Optimization for Minimax Algorithm in Tic-Tac-Toe with Variable Boar  [python]
  0.154  Is there a way to create a variable that will be accessible by all futures from   [python]
  0.150  Why does my code not remove letters from available letter  [python]
  0.147  tic tac toe Using minMax algorithm  [python]
  0.146  Minimax function for board game  [python]
QUERY: Why does Python return True when comparing two datetime objects with different UTC timesta
  0.336  Why are python timezone.utc and ZoneInfo("UTC") not equivalent tzinfo objects? H  [python]
  0.272  Python datetime and timedelta behaviour over Daylight savings time change  [python]
  0.271  Inconsistencies in Time Arithmetic with Timezone-Aware DateTime Objects in Pytho  [python]
  0.264  python: Incorrect(?) time difference b

## 5b. Candidate B — sentence embeddings (all-MiniLM-L6-v2) + cosine
Encodes catalog and queries with a pretrained embedding model; same evaluation protocol as
TF-IDF (queries = first `max_eval_queries` test rows, catalog = train). Also encodes the full
catalog (`E_all`) so the deployed app can load precomputed embeddings.


In [17]:
from sentence_transformers import SentenceTransformer

EMB_MODEL = "all-MiniLM-L6-v2"
emb_model = SentenceTransformer(EMB_MODEL)

def evaluate_sims(sims, queries, catalog, k):
    """precision@k / recall@k / MRR for a similarity matrix (n_queries x n_catalog)."""
    qtags = queries["domain_tag_list"].tolist()
    ctags = catalog["domain_tag_list"].tolist()
    prec, rec, rr = [], [], []
    for i in range(len(queries)):
        top = np.argsort(-sims[i])[:k]
        true_tags = set(qtags[i])
        hits = sum(1 for j in top if set(ctags[j]) & true_tags)
        n_rel = sum(1 for tags in ctags if set(tags) & true_tags)
        prec.append(hits / k)
        rec.append(hits / max(1, n_rel))
        for rank, j in enumerate(top, start=1):
            if set(ctags[j]) & true_tags:
                rr.append(1 / rank)
                break
        else:
            rr.append(0.0)
    return float(np.mean(prec)), float(np.mean(rec)), float(np.mean(rr))

# embeddings: train catalog, test queries, val queries (for tuning), full catalog (for the app)
E_train = emb_model.encode(train["text_all"].tolist(), convert_to_numpy=True, show_progress_bar=True)
E_test  = emb_model.encode(test.iloc[:n_eval]["text_all"].tolist(), convert_to_numpy=True, show_progress_bar=True)
E_val   = emb_model.encode(val["text_all"].tolist(), convert_to_numpy=True, show_progress_bar=True)
E_all   = emb_model.encode(df["text_all"].tolist(), convert_to_numpy=True, show_progress_bar=True)

emb_sims = cosine_similarity(E_test, E_train)

MODEL_RESULTS = {}
for k in CONFIG["top_k"]:
    p, r, mrr = evaluate_sims(sims, test.iloc[:n_eval], train, k)  # TF-IDF sims from cell 11
    MODEL_RESULTS.setdefault("tfidf", {})[k] = (p, r, mrr)
    log(f"TF-IDF      @{k}: precision={p:.3f} recall={r:.3f} MRR={mrr:.3f}")
for k in CONFIG["top_k"]:
    p, r, mrr = evaluate_sims(emb_sims, test.iloc[:n_eval], train, k)
    MODEL_RESULTS.setdefault("embeddings", {})[k] = (p, r, mrr)
    log(f"embeddings  @{k}: precision={p:.3f} recall={r:.3f} MRR={mrr:.3f}")

/Users/bipin/Documents/aberdeen msc ai/applied_artificial_intelligence/assessment_1/assessment_1_task_1/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/bipin/Documents/aberdeen msc ai/applied_artificial_intelligence/assessment_1/assessment_1_task_1/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 1128/1128 [01:17<00:00, 14.51it/s]
/Users/bipin/Documents/aberdeen msc ai/applied_artificial_intelligence/assessment_1/assessment_1_task_1/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/b

TF-IDF      @5: precision=0.981 recall=0.000 MRR=0.988
TF-IDF      @10: precision=0.976 recall=0.000 MRR=0.988
embeddings  @5: precision=0.990 recall=0.000 MRR=0.994
embeddings  @10: precision=0.989 recall=0.000 MRR=0.994


## 5c. Model comparison — pick the deployed model
Summary of all candidates on the test split; the best by precision@5 is recorded in
`model_selection.json` and the Streamlit app loads that model.


In [19]:
log("=== model comparison (test split) ===")
for name, res in MODEL_RESULTS.items():
    for k in sorted(res):
        p, r, mrr = res[k]
        log(f"{name:10s} @{k}: precision={p:.3f} recall={r:.3f} MRR={mrr:.3f}")
best_model = max(MODEL_RESULTS, key=lambda n: MODEL_RESULTS[n][5][0])
log(f"best model by precision@5: {best_model}")

=== model comparison (test split) ===
tfidf      @5: precision=0.981 recall=0.000 MRR=0.988
tfidf      @10: precision=0.976 recall=0.000 MRR=0.988
embeddings @5: precision=0.990 recall=0.000 MRR=0.994
embeddings @10: precision=0.989 recall=0.000 MRR=0.994
hybrid     @5: precision=0.990 recall=0.000 MRR=0.994
hybrid     @10: precision=0.989 recall=0.000 MRR=0.994
best model by precision@5: embeddings


## 6. Persist model artifacts (what the Streamlit app loads from `outputs/`)


In [ ]:
from scipy.sparse import save_npz
import joblib

out = CONFIG["output_dir"]
save_npz(f"{out}/X_train.npz", X_train)
save_npz(f"{out}/X_val.npz", X_val)
save_npz(f"{out}/X_test.npz", X_test)
np.savez_compressed(f"{out}/Y.npz", Y=Y)
json.dump({"tag2idx": tag2idx}, open(f"{out}/tag_vocab.json", "w"), indent=1)
CONFIG["tfidf"] = {k: (list(v) if isinstance(v, tuple) else v) for k, v in best_tfidf_params.items()}
json.dump(CONFIG, open(f"{out}/config.json", "w"), indent=1, default=str)
joblib.dump(vectorizer, f"{out}/vectorizer.joblib")
np.savez_compressed(f"{out}/embeddings_all.npz", E=E_all)
json.dump({"emb_model": EMB_MODEL,
           "model_results": {n: {str(k): list(v) for k, v in r.items()} for n, r in MODEL_RESULTS.items()}},
          open(f"{out}/model_selection.json", "w"), indent=1)
open(f"{out}/training_log.txt", "w").write("\n".join(LOG))
log(f"saved model artifacts to {out}/")
